In [0]:
%sql
DROP TABLE IF EXISTS workspace.nba_silver.dim_team;
DROP TABLE IF EXISTS workspace.nba_silver.fact_game;


In [0]:
# =====================================================================================
# Notebook: 02_silver/01_build_silver_tables.py
# Purpose : Build Silver tables (serverless + UC safe)
#           - dim_team
#           - fact_game
#           - fact_game_enriched
#
# Robustness:
# - In Bronze, columns are expected: dt, meta, data
# - Sometimes meta/data end up as STRING (JSON)
# - Serverless disallows RDD-based schema inference
# - We normalize meta and data using explicit schemas (best practice).
# =====================================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, BooleanType
)

# -----------------------------
# 0) Table names
# -----------------------------
BRONZE_TEAMS = "workspace.nba_bronze.balldontlie_teams"
BRONZE_GAMES = "workspace.nba_bronze.balldontlie_games"

SILVER_DIM_TEAM = "workspace.nba_silver.dim_team"
SILVER_FACT_GAME = "workspace.nba_silver.fact_game"
SILVER_FACT_GAME_ENRICHED = "workspace.nba_silver.fact_game_enriched"

print("BRONZE_TEAMS =", BRONZE_TEAMS)
print("BRONZE_GAMES =", BRONZE_GAMES)

# -----------------------------
# 1) Schemas for parsing Bronze JSON strings
# -----------------------------
META_SCHEMA = StructType([
    StructField("dt", StringType(), True),
    StructField("entity", StringType(), True),
    StructField("ingested_at", StringType(), True),
    StructField("request_url", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("source", StringType(), True),
    StructField("status_code", LongType(), True),
])

TEAM_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("abbreviation", StringType(), True),
    StructField("city", StringType(), True),
    StructField("conference", StringType(), True),
    StructField("division", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("name", StringType(), True),
])

GAME_TEAM_SCHEMA = TEAM_SCHEMA

GAME_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("season", LongType(), True),
    StructField("status", StringType(), True),
    StructField("date", StringType(), True),
    StructField("datetime", StringType(), True),

    StructField("home_team_score", LongType(), True),
    StructField("visitor_team_score", LongType(), True),

    StructField("home_team", GAME_TEAM_SCHEMA, True),
    StructField("visitor_team", GAME_TEAM_SCHEMA, True),

    StructField("postseason", BooleanType(), True),
    StructField("period", LongType(), True),
    StructField("time", StringType(), True),
])

# -----------------------------
# 2) Helper: normalize JSON struct columns (meta + data)
# -----------------------------
def normalize_json_struct(df, colname, schema):
    if colname not in df.columns:
        raise ValueError(f"Expected column '{colname}' not found")

    dtype = dict(df.dtypes).get(colname)
    if dtype is None:
        raise ValueError(f"Could not determine dtype for '{colname}'")

    if dtype.startswith("struct"):
        return df

    if dtype == "string":
        return df.withColumn(colname, F.from_json(F.col(colname), schema))

    raise ValueError(f"Unsupported dtype for {colname}: {dtype}")

# -----------------------------
# 3) Helper: safe ingested_at
# -----------------------------
def _safe_ingested_at(df):
    return F.coalesce(
        F.to_timestamp(F.col("meta.ingested_at")),
        F.to_timestamp(F.col("_loaded_at")),
        F.current_timestamp()
    )

# -----------------------------
# 4) Load Bronze + normalize meta/data
# -----------------------------
teams_bronze = spark.table(BRONZE_TEAMS)
games_bronze = spark.table(BRONZE_GAMES)

teams_bronze = normalize_json_struct(teams_bronze, "meta", META_SCHEMA)
games_bronze = normalize_json_struct(games_bronze, "meta", META_SCHEMA)

teams_bronze = normalize_json_struct(teams_bronze, "data", TEAM_SCHEMA)
games_bronze = normalize_json_struct(games_bronze, "data", GAME_SCHEMA)

print("teams_bronze rows =", teams_bronze.count())
print("games_bronze rows =", games_bronze.count())

# =====================================================================================
# 5) dim_team
# =====================================================================================
dim_team_raw = (
    teams_bronze
    .select(
        F.col("data.id").cast("int").alias("team_id_bdl"),
        F.trim(F.col("data.abbreviation")).alias("abbreviation"),
        F.trim(F.col("data.city")).alias("city"),
        F.trim(F.col("data.conference")).alias("conference"),
        F.trim(F.col("data.division")).alias("division"),
        F.trim(F.col("data.full_name")).alias("full_name"),
        F.trim(F.col("data.name")).alias("name"),
        F.col("dt").cast("date").alias("effective_dt"),
        _safe_ingested_at(teams_bronze).alias("ingested_at"),
        F.lit("balldontlie").alias("source_system"),
    )
    .filter(F.col("team_id_bdl").isNotNull())
)

w_team = Window.partitionBy("team_id_bdl").orderBy(F.col("ingested_at").desc())
dim_team = (
    dim_team_raw
    .withColumn("_rn", F.row_number().over(w_team))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

spark.sql(f"DROP TABLE IF EXISTS {SILVER_DIM_TEAM}")
(dim_team.write.format("delta").mode("overwrite").saveAsTable(SILVER_DIM_TEAM))
print("Wrote:", SILVER_DIM_TEAM, "rows=", dim_team.count())

# =====================================================================================
# 6) fact_game
# =====================================================================================
fact_game_raw = (
    games_bronze
    .select(
        F.col("data.id").cast("int").alias("game_id_bdl"),

        F.coalesce(
            F.to_timestamp(F.col("data.datetime")),
            F.to_timestamp(F.col("data.date"))
        ).alias("start_time_utc"),

        F.to_date(
            F.coalesce(
                F.to_timestamp(F.col("data.datetime")),
                F.to_timestamp(F.col("data.date"))
            )
        ).alias("game_date_utc"),

        F.col("data.season").cast("int").alias("season_start_year"),
        F.trim(F.col("data.status")).alias("status"),

        F.col("data.home_team.id").cast("int").alias("home_team_id_bdl"),
        F.col("data.visitor_team.id").cast("int").alias("visitor_team_id_bdl"),

        F.col("data.home_team_score").cast("int").alias("home_score_raw"),
        F.col("data.visitor_team_score").cast("int").alias("visitor_score_raw"),

        F.col("dt").cast("date").alias("source_dt"),
        _safe_ingested_at(games_bronze).alias("ingested_at"),
        F.lit("balldontlie").alias("source_system"),
    )
    .filter(F.col("game_id_bdl").isNotNull())
)

fact_game = (
    fact_game_raw
    .withColumn("home_score", F.when(F.col("home_score_raw") >= 0, F.col("home_score_raw")).otherwise(F.lit(None).cast("int")))
    .withColumn("visitor_score", F.when(F.col("visitor_score_raw") >= 0, F.col("visitor_score_raw")).otherwise(F.lit(None).cast("int")))
    .drop("home_score_raw", "visitor_score_raw")
)

w_game = Window.partitionBy("game_id_bdl").orderBy(F.col("ingested_at").desc())
fact_game = (
    fact_game
    .withColumn("_rn", F.row_number().over(w_game))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

spark.sql(f"DROP TABLE IF EXISTS {SILVER_FACT_GAME}")
(fact_game.write.format("delta").mode("overwrite").partitionBy("source_dt").saveAsTable(SILVER_FACT_GAME))
print("Wrote:", SILVER_FACT_GAME, "rows=", fact_game.count())

display(
    fact_game.orderBy(F.col("start_time_utc").desc())
    .select("game_id_bdl","season_start_year","game_date_utc","start_time_utc","status","source_dt","home_score","visitor_score")
    .limit(25)
)

# =====================================================================================
# 7) fact_game_enriched
# =====================================================================================
dim_team_s = spark.table(SILVER_DIM_TEAM)

home_team = dim_team_s.select(
    F.col("team_id_bdl").alias("home_team_id_bdl_join"),
    F.col("full_name").alias("home_team_name")
)

visitor_team = dim_team_s.select(
    F.col("team_id_bdl").alias("visitor_team_id_bdl_join"),
    F.col("full_name").alias("visitor_team_name")
)

fact_game_enriched = (
    fact_game.alias("g")
    .join(home_team.alias("h"), F.col("g.home_team_id_bdl") == F.col("h.home_team_id_bdl_join"), "left")
    .join(visitor_team.alias("v"), F.col("g.visitor_team_id_bdl") == F.col("v.visitor_team_id_bdl_join"), "left")
    .drop("home_team_id_bdl_join", "visitor_team_id_bdl_join")
)

spark.sql(f"DROP TABLE IF EXISTS {SILVER_FACT_GAME_ENRICHED}")
(fact_game_enriched.write.format("delta").mode("overwrite").partitionBy("source_dt").saveAsTable(SILVER_FACT_GAME_ENRICHED))
print("Wrote:", SILVER_FACT_GAME_ENRICHED, "rows=", fact_game_enriched.count())

display(
    fact_game_enriched.orderBy(F.col("start_time_utc").desc())
    .select("game_id_bdl","start_time_utc","home_team_name","visitor_team_name","home_score","visitor_score","status","source_dt")
    .limit(25)
)